In [1]:
# Dataset
import numpy as np
np.random.seed(42)

X = np.random.uniform(-2,2, (400,3))

y = (np.sin(X[:,0]) + 0.5* (X[:,1]**2) - 0.8*X[:,2])
y = y.reshape(-1,1)

Parameters calculation:

1. Model A (shallow): 3 -> 4 -> 1
    * params = 16 + 5 = 21

2. Model B (medium): 3 -> 6 -> 6 -> 1
    * params = 24 + 42 + 7 = 73

3. Model C (deep): 3 -> 8 -> 8 -> 8-> 8 -> 1
    * params = 32 + 72 + 72 + 72 + 9 = 257

4. Model D (very deep): 3 -> 8 -> 8 -> 8 -> 8 -> 8 -> 8 ->8 ->8 -> 1
    * params = 32 + 7*72 + 9 = 545

In [3]:
# Activation fxn & derivatives
# 1. relu
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return np.where(z > 0, 1, 0)

# 2. sigmoid
def sigmoid(z):
    return 1/(1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1-s)


# 3. tanh
def tanh(z):
    return np.tanh(z) 

def tanh_derivative(z):
    return 1.0 - np.tanh(z)**2


# 4. Leaky ReLU
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)


# 5. Softplus
def softplus(z):
    return np.log(1 + np.exp(z))

def softplus_derivative(z):
    return sigmoid(z)

In [4]:
def initialize_parameters(layer_dims):
    np.random.seed(42)
    parameters = {}
    L = len(layer_dims)

    for l in range(1, L):
        parameters['w'+ str(l)] = np.random.uniform(-0.5, 0.5, (layer_dims[l], layer_dims[l-1]))
        parameters['b' + str(l)] = np.zeros((layer_dims[l], 1))
        
    return parameters

# Model B: 3 -> 4 -> 1
params = initialize_parameters([3, 4, 1])
params

{'w1': array([[-0.12545988,  0.45071431,  0.23199394],
        [ 0.09865848, -0.34398136, -0.34400548],
        [-0.44191639,  0.36617615,  0.10111501],
        [ 0.20807258, -0.47941551,  0.46990985]]),
 'b1': array([[0.],
        [0.],
        [0.],
        [0.]]),
 'w2': array([[ 0.33244264, -0.28766089, -0.31817503, -0.31659549]]),
 'b2': array([[0.]])}

Forward & backward prop

In [5]:
def forward_propagation(X, parameters, hidden_activation):
    caches = []
    A = X
    L = len(parameters) // 2 
    
    for l in range(1, L):
        A_prev = A
        W = parameters['w'+ str(l)]
        b = parameters['b' + str(l)]
        
        Z = np.dot(W, A_prev) + b
        
        # A = activation(Z)
        if hidden_activation == 'relu':
            A = relu(Z)
        elif hidden_activation == 'sigmoid':
            A = sigmoid(Z)
            
        caches.append((A_prev, Z, W))
    
    W = parameters['w'+ str(L)]
    b = parameters['b' + str(L)]
    Z = np.dot(W, A) + b
    AL = Z
    
    caches.append((A, Z, W))
    
    return AL, caches

In [6]:
def backward_propagation(AL, Y, caches, hidden_activation):
    grads = {}
    L = len(caches)
    N = AL.shape[1]

    dAL = 2 * (AL - Y)
    
    dZ = dAL
    
    for l in reversed(range(1, L + 1)):
        A_prev, Z, W = caches[l-1]
        
        grads['dW' + str(l)] = (1 / N) * np.dot(dZ, A_prev.T)
        grads['db' + str(l)] = (1 / N) * np.sum(dZ, axis=1, keepdims=True)
        
        if l > 1:
            dA_prev = np.dot(W.T, dZ)
            
            _, Z_prev, _ = caches[l-2]
            
            if hidden_activation == 'relu':
                dZ = dA_prev * relu_derivative(Z_prev)
            elif hidden_activation == 'sigmoid':
                dZ = dA_prev * sigmoid_derivative(Z_prev)
                
    return grads

Param update

In [7]:
import numpy as np

def update_parameters(parameters, grads, learning_rate):
    L = len(parameters) // 2
    
    for l in range(1, L + 1):
        parameters['w'+ str(l)] -= learning_rate * grads['dW' + str(l)]
        parameters['b' + str(l)] -= learning_rate * grads['db' + str(l)]
        
    return parameters

def compute_gradient_norm(G):
    return np.sqrt(np.sum(np.square(G)))

Train model

In [8]:
def train_model(X, Y, layer_dims, hidden_activation, epochs=1000, learning_rate=0.01):
    parameters = initialize_parameters(layer_dims)
    L = len(layer_dims) - 1
    
    loss_200 = None
    
    for epoch in range(1, epochs + 1):
        # 1. Forward pass
        AL, caches = forward_propagation(X, parameters, hidden_activation)
        
        # 2. Compute MSE 
        n = Y.shape[1]
        loss = (1.0 / n) * np.sum(np.square(Y - AL))
        
        # 3. Backward pass
        grads = backward_propagation(AL, Y, caches, hidden_activation)
        
        # 4. Update params
        parameters = update_parameters(parameters, grads, learning_rate)
        
        if epoch == 200:
            loss_200 = loss
            
        if epoch == epochs:
            final_loss = loss
            
            grad_norm_L1 = compute_gradient_norm(grads['dW1'])
            
            last_hidden_idx = L - 1 if (L - 1) > 0 else 1 
            grad_norm_last = compute_gradient_norm(grads['dW' + str(last_hidden_idx)])
            
    return parameters, final_loss, grad_norm_L1, grad_norm_last

In [9]:
X_train = X.T
Y_train = y.T

models = [
    {"name": "Model A (Shallow)", "dims": [3, 4,1],"activation": "relu"},
    {"name": "Model B (Medium)", "dims": [3, 6, 6, 1], "activation": "relu"}, 
    {"name": "Model C (Deep)", "dims": [3, 8, 8, 8, 8, 1], "activation": "relu"},
    {"name": "Model D (Very Deep)", "dims": [3, 8, 8, 8, 8, 8, 8, 8, 8, 1], "activation": "relu"},
    {"name": "Model D (Very Deep)", "dims": [3, 8, 8, 8, 8, 8, 8, 8, 8, 1], "activation": "sigmoid"}
]

print(f"{'Model Name':<25} | {'Activation':<10} | {'Final Loss':<12} | {'Grad Norm L1':<12} | {'Grad Norm Last':<12}")
print("-" * 80)

for config in models:
    _, final_loss, grad_norm_L1, grad_norm_last = train_model(
        X_train, 
        Y_train, 
        layer_dims=config["dims"], 
        hidden_activation=config["activation"],
        epochs=1000, 
        learning_rate=0.01
    )
    print(f"{config['name']:<25} | {config['activation'].upper():<10} | {final_loss:<12.4f} | {grad_norm_L1:<12.6f} | {grad_norm_last:<12.6f}")

print("-" * 80)

Model Name                | Activation | Final Loss   | Grad Norm L1 | Grad Norm Last
--------------------------------------------------------------------------------
Model A (Shallow)         | RELU       | 0.0402       | 0.036178     | 0.036178    
Model B (Medium)          | RELU       | 0.0563       | 0.066831     | 0.029547    
Model C (Deep)            | RELU       | 0.0521       | 0.018136     | 0.008198    
Model D (Very Deep)       | RELU       | 0.0875       | 0.758373     | 0.959874    
Model D (Very Deep)       | SIGMOID    | 1.7439       | 0.000006     | 0.000008    
--------------------------------------------------------------------------------


Reflections

1. Did deeper always reduce loss faster?
    * No, adding more layers actually made the network's final error worse instead of better

2. Did gradients in early layers stay similar to later layers?
    * No, either go down to almost nothing or increase larger as going deep layers

3. Was training equally stable for all activations?
    * No, relu managed to keep working in the deepest network, while sigmoid failed to learn

4. Which activation behaved more stable in deeper networks?
    * Relu was more stable because it allowed the deep network to reduce its error

5. Did some models improve very slowly even though learning rate was same?
    * Yes, deep sigmoid model got completely stuck and barely improved because its learning goes to near zero